## Installation


In [ ]:
%pip install "gymnasium[classic-control]"

## 2. Imports
- `gymnasium` — the RL environment toolkit itself
- `numpy` — for arrays and numerical operations (Gymnasium observations come back as NumPy arrays)
- `matplotlib.pyplot` — for plotting our agent's learning progress, and for building our video clips


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

SEED = 0
np.random.seed(SEED)

## Creating the Environment
We create environments with `gym.make(environment_id)`. The id is just a string Gymnasium looks up in its registry — `"Acrobot-v1"` here.


In [ ]:
env = gym.make("Acrobot-v1")
env

`env` is now an object representing the Acrobot world. We haven't started an episode yet — for that, we need `reset()`.

### Python refresher: tuple unpacking
`env.reset()` returns **two things at once**: the starting observation, and an `info` dictionary (usually empty, used for debugging/extra info). In Python, when a function returns multiple values, you can "unpack" them directly into separate variables in one line:

```python
observation, info = env.reset()
```

This is the exact same trick as:
```python
x, y = 3, 4
```
Python just matches up the items on the right with the names on the left, in order.


In [ ]:
observation, info = env.reset(seed=SEED)

print("Observation:", observation)
print("Info:", info)

In [ ]:
print("Observation space:", env.observation_space)
print("Shape:", env.observation_space.shape)
print("Lower bounds:", env.observation_space.low)
print("Upper bounds:", env.observation_space.high)


In [ ]:
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)


# .sample() picks a uniformly random valid action
print("A random action:", env.action_space.sample())

In [ ]:
observation, info = env.reset(seed=SEED)

action = env.action_space.sample()  # pick a random action
observation, reward, terminated, truncated, info = env.step(action)

print("Action taken:", action)
print("New observation:", observation)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)


In [ ]:
observation, info = env.reset(seed=SEED)
total_reward = 0
steps = 0

while True:
    action = env.action_space.sample()                     # random action
    observation, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    steps += 1

    done = terminated or truncated
    if done:
        break

print(f"Episode finished after {steps} steps.")
print(f"Total reward: {total_reward}")
print(f"Succeeded (terminated early)? {terminated}")

In [ ]:
from IPython.display import HTML
from matplotlib import animation


def collect_random_episode_frames(seed=SEED, max_steps=500):
    """Run one episode with random actions and return the list of rendered frames."""
    render_env = gym.make("Acrobot-v1", render_mode="rgb_array")
    obs, info = render_env.reset(seed=seed)

    frames = [render_env.render()]
    for _ in range(max_steps):
        action = render_env.action_space.sample()
        _, _, terminated, truncated, _ = render_env.step(action)
        frames.append(render_env.render())
        if terminated or truncated:
            break

    render_env.close()
    return frames

def make_animation(frames, title=""):
    """Turn a list of image frames into an inline, playable animation."""
    fig, ax = plt.subplots()
    ax.axis("off")
    if title:
        ax.set_title(title)
    img = ax.imshow(frames[0])

    def update(i):
        img.set_data(frames[i])
        return [img]

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=40, blit=True
    )
    plt.close(fig)  # prevents a duplicate static image from also being displayed
    return anim

random_frames = collect_random_episode_frames()
print(f"Collected {len(random_frames)} frames from the random-action episode.")

random_anim = make_animation(random_frames, title="Random actions")
HTML(random_anim.to_jshtml())

In [ ]:
# Number of bins per dimension. Keep this modest -- the table size grows
# multiplicatively! With 6 dimensions, 6 bins each gives 6**6 ~ 46,656 states.
N_BINS = 6

obs_low = env.observation_space.low
obs_high = env.observation_space.high

# Angular velocities are technically unbounded by physics but Gymnasium
# documents a practical range (+-4*pi and +-9*pi); we use those bounds directly.
print("Low:", obs_low)
print("High:", obs_high)

# Build one set of bin edges per dimension.
# We use N_BINS - 1 edges, which creates N_BINS bins (think: 3 fence posts make 2 fields,
# but here we use a fencepost convention that yields exactly N_BINS bins -- see check below).
bin_edges = [
    np.linspace(obs_low[i], obs_high[i], N_BINS - 1)
    for i in range(len(obs_low))
]

def discretize(observation):
    """Convert a continuous 6-value observation into a tuple of 6 bin indices."""
    return tuple(
        int(np.digitize(observation[i], bin_edges[i]))
        for i in range(len(observation))
    )

# Quick sanity check
obs, info = env.reset(seed=SEED)
print("Raw observation:", obs)
print("Discretized state:", discretize(obs))


In [ ]:
Q = {}  # maps discretized_state -> array of Q-values, one per action


def get_q_values(state):
    """Return the Q-values for a state, creating a fresh all-zero row if we haven't seen it before."""
    if state not in Q:
        Q[state] = np.zeros(env.action_space.n)
    return Q[state]


def epsilon_greedy_action(state, epsilon):
    if np.random.rand() < epsilon:
        return env.action_space.sample()  # explore
    else:
        return int(np.argmax(get_q_values(state)))  # exploit


## Hyperparameters
Four knobs control how training behaves:
- **alpha** - learning rate: how much each new experience updates our existing estimate
- **gamma** - discount factor: how much we value future rewards vs. immediate ones
- **epsilon** - exploration rate, starting high and decaying over time
- **episodes** - how many practice rounds to run



In [ ]:
alpha = 0.1  # learning rate
gamma = 0.99  # discount factor

epsilon = 1.0  # start by exploring 100% of the time
epsilon_min = 0.05  # never fully stop exploring
epsilon_decay = 0.9995  # multiply epsilon by this after every episode

episodes = 5000
max_steps_per_episode = 500  # matches Acrobot's own truncation limit

# Tracking, for plotting later
episode_rewards = []


In [ ]:
for ep in range(episodes):
    obs, info = env.reset()
    state = discretize(obs)
    total_reward = 0

    for t in range(max_steps_per_episode):
        action = epsilon_greedy_action(state, epsilon)

        next_obs, reward, terminated, truncated, info = env.step(action)
        next_state = discretize(next_obs)
        done = terminated or truncated

        # Q-learning update
        best_next_value = np.max(get_q_values(next_state))
        td_target = reward + gamma * best_next_value
        td_error = td_target - get_q_values(state)[action]
        Q[state][action] += alpha * td_error

        state = next_state
        total_reward += reward

        if done:
            break

    episode_rewards.append(total_reward)

    # Decay epsilon, but never below epsilon_min
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if (ep + 1) % 500 == 0:
        recent_avg = np.mean(episode_rewards[-500:])
        print(f"Episode {ep + 1}/{episodes} | epsilon={epsilon:.3f} | avg reward (last 500): {recent_avg:.1f}")

In [ ]:
def moving_average(x, window=100):
    x = np.array(x, dtype=float)
    if len(x) < window:
        return x
    cumsum = np.cumsum(np.insert(x, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / float(window)

smoothed = moving_average(episode_rewards, window=100)

plt.figure(figsize=(8, 4))
plt.plot(episode_rewards, alpha=0.2, label="raw reward")
plt.plot(smoothed, color="orange", label="moving average (window=100)")
plt.title("Acrobot: Reward per Episode")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
def evaluate_policy(env, Q, episodes=20, max_steps=500):
    rewards = []
    successes = 0
    for _ in range(episodes):
        obs, _ = env.reset()
        state = discretize(obs)
        total_reward = 0
        for _ in range(max_steps):
            action = int(np.argmax(get_q_values(state)))
            obs, reward, terminated, truncated, _ = env.step(action)
            state = discretize(obs)
            total_reward += reward
            if terminated or truncated:
                if terminated:
                    successes += 1
                break
        rewards.append(total_reward)
    return np.mean(rewards), successes / episodes

eval_env = gym.make("Acrobot-v1")
avg_reward, success_rate = evaluate_policy(eval_env, Q)
print(f"Average reward over evaluation episodes: {avg_reward:.1f}")
print(f"Success rate (reached the target height before timing out): {success_rate * 100:.0f}%")

## Watching the Trained Agent
Now let's make the same kind of clip as before, but using the **greedy** policy instead of random actions. We can reuse our `make_animation()` helper from before; only the action-selection logic changes.


In [ ]:
def collect_trained_episode_frames(Q, seed=SEED, max_steps=500):
    """Run one episode using the greedy (trained) policy and return the rendered frames."""
    render_env = gym.make("Acrobot-v1", render_mode="rgb_array")
    obs, info = render_env.reset(seed=seed)
    state = discretize(obs)

    frames = [render_env.render()]
    for _ in range(max_steps):
        action = int(np.argmax(get_q_values(state)))
        obs, _, terminated, truncated, _ = render_env.step(action)
        state = discretize(obs)
        frames.append(render_env.render())
        if terminated or truncated:
            break

    render_env.close()
    return frames